# ¿Sirve φ⁴ como medidor de riesgo? Validación como alerta temprana

**Pregunta:** ¿los indicadores del φ⁴ calculados hoy anticipan la turbulencia de las próximas 4 semanas mejor que
lo que ya usa la industria?

**Universo:** las 20 mayores acciones del S&P 500 (las mismas de `sp500_top20_riesgo.ipynb`), 10 años. La
turbulencia se mide en un portafolio equiponderado de las 20.

| Tipo | Indicador (ventana de 120 días hasta la fecha t) |
| --- | --- |
| **φ⁴** | acoplamiento medio w/√(μμ) · fuerza del mayor hub · dispersión de log μ_i · curtosis transversal del modelo |
| Referencia de la literatura | correlación media · *absorption ratio* (Kritzman et al., 2011; varianza absorbida por los 4 primeros vectores propios) |
| **Vara mínima** | volatilidad actual del portafolio (últimos 20 días) |

**Objetivos (t+1 a t+20):** volatilidad realizada y caída máxima del portafolio; "evento de estrés" = el 10% de las
fechas con mayor caída siguiente.

**Por qué la volatilidad actual es la vara mínima:** la volatilidad se agrupa en el tiempo, así que la de hoy ya
anticipa buena parte de la de mañana. Un indicador solo sirve si agrega información **además** de ella.

**Cómo se decide:**
1. **Dentro de muestra:** regresión objetivo ~ volatilidad actual + indicador, con errores de Newey–West (los
   objetivos se solapan).
2. **Fuera de muestra (la prueba que importa):** regresión con ventana expansiva, sin que el objetivo de
   entrenamiento se solape con la fecha evaluada. R²_oos > 0 significa que el indicador mejora al modelo base;
   la prueba de Clark–West dice si la mejora es significativa.
3. Dos modelos base: (a) solo volatilidad actual; (b) volatilidad + correlación media + absorption ratio. Pasar (b)
   es lo que haría del φ⁴ un medidor de riesgo con valor propio.

**Tiempo estimado:** 5–15 minutos (el φ⁴ se reajusta cada `STEP` días). Usa la misma caché de precios que el
notebook del mapa de riesgo.

In [ ]:
import time
import warnings
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy.stats import spearmanr

import phi4finance
from phi4finance import RollingPhi4, load_returns
from phi4finance.earlywarning import (absorption_ratio, auc, average_correlation, forward_max_drawdown,
                                      forward_realized_vol, hac_ols, oos_r2, rolling_indicator,
                                      trailing_realized_vol)

TICKERS = ["NVDA", "AAPL", "MSFT", "AMZN", "GOOGL", "META", "AVGO", "TSLA", "MU", "BRK-B",
           "LLY", "AMD", "JPM", "WMT", "V", "INTC", "XOM", "JNJ", "MA", "ABBV"]
MARKET = "SPY"
TODAY = pd.Timestamp.today().normalize()
START = (TODAY - pd.DateOffset(years=11)).strftime("%Y-%m-%d")
END = (TODAY + pd.Timedelta(days=1)).strftime("%Y-%m-%d")
DATA_DIR, RESULTS_DIR = Path("data"), Path("results")

FULL = False
WIN = 120                       # ventana de los indicadores (días)
STEP = 5 if FULL else 10        # cada cuántos días se calcula (φ⁴ se reajusta)
H = 20                          # horizonte del objetivo (días hábiles)
K_AR = 4                        # vectores propios del absorption ratio (20 acciones / 5)
EVENT_Q = 0.90                  # evento = caída siguiente por encima de su percentil 90
MIN_TRAIN_YEARS = 3             # historia mínima antes de la primera predicción fuera de muestra
print("phi4finance", phi4finance.__version__, "| FULL =", FULL)

In [ ]:
INK, INK2, MUTED, GRID = "#0b0b0b", "#52514e", "#898781", "#e1e0d9"
C_PHI4, C_BENCH, C_BASE, C_TARGET = "#eb6834", "#2a78d6", "#898781", "#0b0b0b"
plt.rcParams.update({
    "figure.facecolor": "#fcfcfb", "axes.facecolor": "#fcfcfb", "savefig.facecolor": "#fcfcfb",
    "axes.edgecolor": "#c3c2b7", "axes.labelcolor": INK2, "axes.titlecolor": INK,
    "axes.titlesize": 11, "axes.titleweight": "bold", "axes.titlelocation": "left",
    "axes.grid": True, "grid.color": GRID, "grid.linewidth": 0.6, "axes.axisbelow": True,
    "axes.spines.top": False, "axes.spines.right": False,
    "xtick.color": MUTED, "ytick.color": MUTED, "xtick.labelcolor": INK2, "ytick.labelcolor": INK2,
    "lines.linewidth": 1.4, "legend.frameon": False, "legend.labelcolor": INK2,
    "font.size": 10, "figure.dpi": 110,
})


def styled(df, fmt):
    try:
        return df.style.format(fmt)
    except (AttributeError, ImportError):
        return df.round(4)

## 1. Datos y portafolio

Retornos simples diarios de cierres ajustados; el portafolio se rebalancea cada día a 5% por acción.

In [ ]:
with warnings.catch_warnings(record=True) as w:
    warnings.simplefilter("always", UserWarning)
    RALL = load_returns(TICKERS + [MARKET], start=START, end=END, log=False, cache_dir=DATA_DIR)
for x in w:
    if issubclass(x.category, UserWarning) and "unclosed" not in str(x.message):
        print("aviso:", x.message)
TICKERS = [t for t in TICKERS if t in RALL.columns]
R = RALL[TICKERS]
R = R.loc[R.index > R.index[-1] - pd.DateOffset(years=10, days=int(WIN * 1.6))]   # 10 años + arranque de la ventana
P = R.mean(axis=1)
print(f"{len(R)} días, {R.index[0].date()} a {R.index[-1].date()}, {R.shape[1]} acciones")

## 2. Indicadores

El φ⁴ se ajusta con `RollingPhi4` sobre los retornos de cada ventana (sin filtro de volatilidad: queremos medir el
estado del mercado tal cual), partiendo de los parámetros de la fecha anterior. De cada ajuste salen:

- **acoplamiento medio**: media de w_ij/√(μ_i μ_j) (correlación parcial media que implica el modelo);
- **fuerza del mayor hub**: la acción más conectada, Σ_j |w_ij|/√(μ_i μ_j);
- **dispersión de log μ_i**: cuán distintas son las volatilidades entre acciones (es lo que genera curtosis
  transversal en el modelo);
- **curtosis del modelo**: curtosis transversal media de 1000 configuraciones muestreadas (el indicador de la Fig. 1
  del paper).

In [ ]:
t0 = time.time()
roll = RollingPhi4(window=WIN, step=STEP, n_samples=1000, burn=200, l2=0.01, verbose=True,
                   model_kw={"mu_global": False, "lam_global": False}, fit_kw={"n_grid": 101})
roll.fit(R)
res = roll.results_
dates = res.index
print(f"{len(res)} ajustes φ⁴ en {time.time() - t0:.0f} s")

IND = pd.DataFrame({
    "φ⁴ acoplamiento medio": res.coupling_mean,
    "φ⁴ mayor hub": res.hub_strength_max,
    "φ⁴ dispersión log μ": res.log_mu_std,
    "φ⁴ curtosis modelo": res.model_market_kurtosis,
    "curtosis datos": res.data_market_kurtosis,
    "correlación media": rolling_indicator(R, average_correlation, WIN, dates),
    "absorption ratio": rolling_indicator(R, lambda x: absorption_ratio(x, K_AR), WIN, dates),
    "vol. actual (20d)": trailing_realized_vol(P, 20).reindex(dates),
})
PHI4 = [c for c in IND.columns if c.startswith("φ⁴")]
BENCH = ["correlación media", "absorption ratio", "curtosis datos"]
print("\nCorrelación entre indicadores (¿el φ⁴ mide algo distinto?):")
IND.corr(method="spearman").round(2)

## 3. Objetivos y evidencia

Para cada fecha t: volatilidad realizada (log) y caída máxima del portafolio en t+1…t+20. Los gráficos muestran cada
indicador y la volatilidad siguiente, ambos estandarizados (media 0, desviación 1) en el mismo eje.

In [ ]:
Y = pd.DataFrame({"log vol. siguiente": np.log(forward_realized_vol(P, H)),
                  "caída máx. siguiente": forward_max_drawdown(P, H)}).reindex(dates)
D = IND.join(Y).dropna()
D["log vol. actual"] = np.log(D["vol. actual (20d)"])
D["evento"] = D["caída máx. siguiente"] > D["caída máx. siguiente"].quantile(EVENT_Q)
print(f"{len(D)} fechas con objetivo, {D.evento.sum()} eventos de estrés ({D.index[0].date()} a {D.index[-1].date()})")

zs = lambda s: (s - s.mean()) / s.std()
show = PHI4 + ["correlación media", "absorption ratio"]
fig, axes = plt.subplots(len(show), 1, figsize=(11, 1.9 * len(show)), sharex=True)
for ax, c in zip(axes, show):
    ax.plot(D.index, zs(D["log vol. siguiente"]), color=C_TARGET, lw=0.9, alpha=0.55, label="log vol. siguiente")
    ax.plot(D.index, zs(D[c]), color=C_PHI4 if c in PHI4 else C_BENCH, lw=1.4, label=c)
    for d in D.index[D.evento]:
        ax.axvspan(d, d + pd.Timedelta(days=STEP * 1.4), color="#e34948", alpha=0.08, lw=0)
    ax.set_title(f"{c}  (ρ Spearman con vol. siguiente = {spearmanr(D[c], D['log vol. siguiente'])[0]:.2f})", fontsize=10)
    ax.legend(loc="upper left", fontsize=7, ncol=2)
axes[-1].set_xlabel("Fecha (bandas rojas: fechas previas a un evento de estrés)")
fig.tight_layout(); plt.show()

In [ ]:
LAGS = int(np.ceil(H / roll.step))
rows = []
for target in ["log vol. siguiente", "caída máx. siguiente"]:
    for c in PHI4 + BENCH:
        X = pd.DataFrame({"log vol. actual": D["log vol. actual"], c: zs(D[c])})
        r = hac_ols(D[target], X, lags=LAGS)
        rows.append({"objetivo": target, "indicador": c, "coef. (por 1σ)": r.loc[c, "coef"],
                     "t (Newey–West)": r.loc[c, "t"], "p": r.loc[c, "p"],
                     "ρ Spearman": spearmanr(D[c], D[target])[0]})
tab_in = pd.DataFrame(rows).set_index(["objetivo", "indicador"])
print("Dentro de muestra: efecto del indicador controlando por la volatilidad actual")
styled(tab_in, {"coef. (por 1σ)": "{:.4f}", "t (Newey–West)": "{:.2f}", "p": "{:.3f}", "ρ Spearman": "{:.2f}"})

## 4. Fuera de muestra (la prueba que decide)

En cada fecha se reestima la regresión con la historia disponible (al menos 3 años), dejando fuera las
observaciones cuyo objetivo se solapa con la fecha evaluada, y se predice el objetivo. Se compara el error con el de
un modelo base:

- **Base A**: solo la volatilidad actual.
- **Base B**: volatilidad actual + correlación media + absorption ratio.

R²_oos > 0 y p (Clark–West) < 0.05 = el indicador aporta información nueva y utilizable.

In [ ]:
MIN_TRAIN = int((D.index < D.index[0] + pd.DateOffset(years=MIN_TRAIN_YEARS)).sum())
GAP = int(np.ceil(H / roll.step))          # filas cuyo objetivo se solapa con la fecha evaluada
print(f"primera predicción fuera de muestra: {D.index[MIN_TRAIN].date()} ({len(D) - MIN_TRAIN} fechas evaluadas)")
baseA = ["log vol. actual"]
baseB = ["log vol. actual", "correlación media", "absorption ratio"]
rows = []
for target in ["log vol. siguiente", "caída máx. siguiente"]:
    y = D[target]
    for c in PHI4 + BENCH:
        for base_name, base in [("A: vol.", baseA), ("B: vol. + corr. + AR", baseB)]:
            if c in base:
                continue
            r = oos_r2(y, D[base], D[base + [c]], MIN_TRAIN, GAP)
            rows.append({"objetivo": target, "indicador": c, "base": base_name,
                         "R² oos": r["r2_oos"], "p (Clark–West)": r["p_cw"], "n": r["n"]})
tab_oos = pd.DataFrame(rows).set_index(["objetivo", "base", "indicador"])
styled(tab_oos, {"R² oos": "{:.3%}", "p (Clark–West)": "{:.3f}", "n": "{:d}"})

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4.8))
for ax, target in zip(axes, ["log vol. siguiente", "caída máx. siguiente"]):
    sub = tab_oos.loc[target].reset_index()
    labels, vals, cols, pv = [], [], [], []
    for _, r in sub.iterrows():
        labels.append(f"{r.indicador}  [{r.base[0]}]")
        vals.append(r["R² oos"]); pv.append(r["p (Clark–West)"])
        cols.append(C_PHI4 if r.indicador in PHI4 else C_BENCH)
    y = np.arange(len(vals))
    ax.barh(y, vals, color=cols, height=0.6)
    ax.axvline(0, color=INK2, lw=1)
    for yi, v, p in zip(y, vals, pv):
        ax.annotate(f"{v:+.1%}{' *' if p < 0.05 else ''}", (v, yi), xytext=(4 if v >= 0 else -4, 0),
                    textcoords="offset points", ha="left" if v >= 0 else "right", va="center", fontsize=7, color=INK2)
    ax.set_yticks(y, labels, fontsize=8); ax.invert_yaxis()
    ax.xaxis.set_major_formatter(plt.matplotlib.ticker.PercentFormatter(1.0))
    ax.set_title(f"R² fuera de muestra: {target}")
fig.text(0.01, 0.01, "Naranja: φ⁴ · azul: referencia · * p < 0.05 (Clark–West) · [A] base = vol. actual · "
         "[B] base = vol. + correlación media + absorption ratio", fontsize=8, color=INK2)
fig.tight_layout(rect=(0, 0.04, 1, 1)); plt.show()

aucs = pd.Series({c: auc(D[c], D.evento) for c in PHI4 + BENCH + ["vol. actual (20d)"]}).sort_values()
fig, ax = plt.subplots(figsize=(8, 3.8))
ax.barh(aucs.index, aucs - 0.5, left=0.5, color=[C_PHI4 if c in PHI4 else (C_BASE if "vol" in c else C_BENCH) for c in aucs.index])
ax.axvline(0.5, color=INK2, lw=1)
for i, v in enumerate(aucs):
    ax.annotate(f"{v:.2f}", (v, i), xytext=(4 if v >= 0.5 else -4, 0), textcoords="offset points",
                ha="left" if v >= 0.5 else "right", va="center", fontsize=8, color=INK2)
ax.set_title("AUC para anticipar eventos de estrés (0.5 = azar)")
fig.tight_layout(); plt.show()

## 5. Veredicto

La celda siguiente escribe el veredicto a partir de los números de tu corrida.

In [ ]:
def verdict(target):
    out = []
    for c in PHI4:
        a = tab_oos.loc[(target, "A: vol.", c)]
        b = tab_oos.loc[(target, "B: vol. + corr. + AR", c)]
        if a["R² oos"] > 0 and a["p (Clark–West)"] < 0.05:
            if b["R² oos"] > 0 and b["p (Clark–West)"] < 0.05:
                msg = "APORTA: mejora a la volatilidad y también a correlación media + absorption ratio"
            else:
                msg = "anticipa, pero no agrega nada sobre correlación media + absorption ratio"
        else:
            msg = "no agrega información sobre la volatilidad actual"
        out.append(f"  {c}: {msg} (R²oos A {a['R² oos']:+.2%}, p {a['p (Clark–West)']:.3f}; "
                   f"B {b['R² oos']:+.2%}, p {b['p (Clark–West)']:.3f})")
    return "\n".join(out)

for target in ["log vol. siguiente", "caída máx. siguiente"]:
    print(f"\nObjetivo: {target}\n{verdict(target)}")
best_ref = tab_oos.xs("A: vol.", level="base").loc[:, "R² oos"].unstack(0)
print("\nReferencia (base A) — correlación media y absorption ratio:")
print(best_ref.loc[["correlación media", "absorption ratio"]].applymap(lambda v: f"{v:+.2%}").to_string()
      if hasattr(best_ref, "applymap") else best_ref)

RESULTS_DIR.mkdir(exist_ok=True)
out = RESULTS_DIR / f"validacion_riesgo_sistemico_{R.index[-1].date()}.xlsx"
try:
    with pd.ExcelWriter(out) as xw:
        tab_oos.to_excel(xw, sheet_name="fuera_de_muestra")
        tab_in.to_excel(xw, sheet_name="dentro_de_muestra")
        aucs.to_frame("AUC").to_excel(xw, sheet_name="auc")
        D.to_excel(xw, sheet_name="serie")
    print("\nguardado en", out)
except ImportError:
    tab_oos.to_csv(out.with_suffix(".csv")); print("\nguardado en", out.with_suffix(".csv"))